In [1]:
# ═══════════════════════════════════════════════
# 1️⃣  MONTER GOOGLE DRIVE & DÉCOMPRESSER LE PROJET
# ═══════════════════════════════════════════════
from google.colab import drive
import zipfile, os, shutil

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Vision_Project.zip'
DEST_DIR = '/content/Vision_Project'

if os.path.isdir(DEST_DIR):
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DEST_DIR)

print('✅ Décompression terminée →', DEST_DIR)

# Vérification de la structure
print('\nContenu de /content/Vision_Project :')
for item in os.listdir(DEST_DIR):
    print(' ', item)

Mounted at /content/drive
✅ Décompression terminée → /content/Vision_Project

Contenu de /content/Vision_Project :
  Vision_Project


In [2]:
# ═══════════════════════════════════════════════
# 2️⃣  INSTALLATION DES DÉPENDANCES
# ═══════════════════════════════════════════════

# Désinstaller facenet-pytorch d'abord (c'est lui qui tire le vieux Pillow)
!pip uninstall -y facenet-pytorch

# Installer facenet-pytorch depuis le source GitHub directement
# (version qui n'a pas de contrainte Pillow stricte)
!pip install -q --no-deps git+https://github.com/timesler/facenet-pytorch.git

# Garder Pillow récent compatible scikit-image + torchvision
!pip install -q "Pillow>=10.1,<11.0" --force-reinstall

# Reste
!pip install -q ultralytics scikit-learn tqdm flask

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
facenet-pytorch 2.5.2 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.0.2 which is incompatible.
facenet-pytorch 2.5.2 requires Pillow<10.3.0,>=10.2.0, but you have pillow 10.4.0 which is incompatible.
facenet-pytorch 2.5.2 requires torch<=2.3.0,>=2.2.0, but you have torch 2.10.0+cu128 which is incompatible.
facenet-pytorch 2.5.2 requires torchvision<=0.18.0,>=0.17.0, but you have torchvision 0.25.0+cu128 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.6 MB/s eta 0:00:00


In [3]:
# ═══════════════════════════════════════════════
# 3️⃣  PYTHONPATH + __init__.py
# ═══════════════════════════════════════════════
import sys, pathlib

PROJECT_ROOT = '/content/Vision_Project/Vision_Project'
sys.path.insert(0, PROJECT_ROOT)  # insert(0) = priorité maximale

# Créer les __init__.py manquants
for pkg in ['securai_store', 'securai_store/modules']:
    init_file = pathlib.Path(PROJECT_ROOT) / pkg / '__init__.py'
    if not init_file.exists():
        init_file.touch()
        print(f'✅ Créé : {init_file}')
    else:
        print(f'ℹ️  Déjà présent : {init_file}')

print('\n✅ PYTHONPATH configuré →', PROJECT_ROOT)

✅ Créé : /content/Vision_Project/Vision_Project/securai_store/__init__.py
ℹ️  Déjà présent : /content/Vision_Project/Vision_Project/securai_store/modules/__init__.py

✅ PYTHONPATH configuré → /content/Vision_Project/Vision_Project


In [4]:
# ═══════════════════════════════════════════════
# 4️⃣  TEST D'IMPORT
# ═══════════════════════════════════════════════
try:
    from securai_store.modules.face_recognizer import FaceRecognizer
    from securai_store.modules.patch_attacker import PatchAttacker
    print('✅ FaceRecognizer importé')
    print('✅ PatchAttacker importé')
except Exception as e:
    print('❌ Import échoué :', e)
    # Diagnostic automatique
    import os
    print('\n── Contenu de /content/Vision_Project :')
    for item in os.listdir('/content/Vision_Project'):
        print(' ', item)

✅ FaceRecognizer importé
✅ PatchAttacker importé


In [5]:
# ═══════════════════════════════════════════════
# 4b️⃣  INSTALLATION CLOUDFLARED
# ═══════════════════════════════════════════════
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O cloudflared.deb
!sudo dpkg -i cloudflared.deb
!cloudflared --version

Selecting previously unselected package cloudflared.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.5.0 (built 2026-05-13-11:24 UTC)


In [1]:
# ═══════════════════════════════════════════════════════════════
# 5️⃣  FLASK API + CLOUDFLARE TUNNEL
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/Vision_Project/Vision_Project')

import subprocess, threading, time
import torch, cv2, base64, numpy as np
from flask import Flask, request, jsonify
from securai_store.modules.face_recognizer import FaceRecognizer
from securai_store.modules.fgsm_attacker import FGSMAttacker

# --- Config ---
PUBLIC_HOSTNAME   = 'vision.api.near-u-api.org'
FLASK_PORT        = 9999
CLOUDFLARED_TOKEN = (
    'eyJhIjoiNDlkMWNmMjU2YTAxNGFiNWVmNmZkYWNiZTExOTRkZmEiLCJ0IjoiODJkOGM4ZTEtZmUx'
    'MC00NzhlLTlhOTctOWUwMWZmZjBlZDBhIiwicyI6Ik1USTFZV1pqWWpndE5ERmpNaTAwTkRreUxU'
    'Z3hZek10WkRBNU5UTTROakJpWldVMyJ9'
)

# --- Libérer le port ---
subprocess.run(['fuser', '-k', f'{FLASK_PORT}/tcp'], capture_output=True)
print(f'[INFO] Port {FLASK_PORT} libéré')

# --- Device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Device: {device}')

# --- Charger les modèles ---
face_rec = FaceRecognizer()
face_rec.model = face_rec.model.to(device)
print('[INFO] FaceRecognizer chargé')

fgsm = FGSMAttacker(face_rec.model, epsilon=0.03)
print('[INFO] FGSMAttacker chargé')

# --- Flask ---
app = Flask(__name__)

def _decode_image(b64_string: str) -> np.ndarray:
    img_bytes = base64.b64decode(b64_string.split(',')[-1])
    np_arr    = np.frombuffer(img_bytes, np.uint8)
    bgr_img   = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)

def _decode_bgr(b64_string: str) -> np.ndarray:
    img_bytes = base64.b64decode(b64_string.split(',')[-1])
    np_arr    = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

# ── Route : reconnaissance faciale ──
@app.route('/infer', methods=['POST'])
def infer():
    try:
        payload = request.get_json()
        if not payload or 'image' not in payload:
            return jsonify({'error': "Missing 'image' field"}), 400
        img_rgb = _decode_image(payload['image'])
        with torch.no_grad():
            name, confidence = face_rec.predict(img_rgb)
        if name in ['Unknown', 'Inconnu']:
            access, name = 'DENIED', 'Inconnu'
        else:
            access = 'GRANTED'
        return jsonify({'name': name, 'confidence': round(float(confidence), 4), 'access': access})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# ── Route : enrôlement depuis app.py local ──
@app.route('/enroll', methods=['POST'])
def enroll_remote():
    try:
        payload = request.get_json()
        name    = payload['name']
        img_bgr = _decode_bgr(payload['image'])
        face_rec.enroll_face(name, img_bgr)
        enrolled = list(face_rec.enrolled_embeddings.keys())
        print(f'[COLAB] Enrôlé : {name} | Total : {enrolled}')
        return jsonify({'success': True, 'enrolled': enrolled})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 400

# ── Route : attaque FGSM sur GPU ──
@app.route('/fgsm', methods=['POST'])
def fgsm_attack():
    try:
        payload     = request.get_json()
        img_bgr     = _decode_bgr(payload['image'])
        target_name = payload.get('target', 'Manager_Demo')
        target_emb  = face_rec.enrolled_embeddings.get(target_name)

        # Calcul FGSM sur GPU
        attacked = fgsm.attack(img_bgr, target_emb)

        _, buf = cv2.imencode('.jpg', attacked, [cv2.IMWRITE_JPEG_QUALITY, 85])
        b64    = "data:image/jpeg;base64," + base64.b64encode(buf).decode()

        # Reconnaissance sur l'image attaquée pour vérifier
        attacked_rgb       = cv2.cvtColor(attacked, cv2.COLOR_BGR2RGB)
        with torch.no_grad():
            name, confidence = face_rec.predict(attacked_rgb)

        print(f'[FGSM] target={target_name} → reconnu={name} conf={confidence:.2f}')
        return jsonify({
            'success':        True,
            'attacked_image': b64,
            'recognized_as':  name,
            'confidence':     round(float(confidence), 4)
        })
    except Exception as e:
        print(f'[FGSM ERREUR] {e}')
        return jsonify({'success': False, 'error': str(e)}), 400

# ── Route : health check ──
@app.route('/health', methods=['GET'])
def health():
    enrolled = list(face_rec.enrolled_embeddings.keys())
    return jsonify({'status': 'ok', 'device': device, 'enrolled': enrolled})

# --- Lancer Flask ---
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(2)
print(f'[INFO] Flask démarré sur le port {FLASK_PORT}')

# --- Cloudflare Tunnel ---
def _run_cloudflared():
    cmd = ['cloudflared', 'tunnel', 'run',
           '--url', f'http://localhost:{FLASK_PORT}',
           '--token', CLOUDFLARED_TOKEN]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    globals()['_cf_proc'] = proc
    for line in iter(proc.stdout.readline, ''):
        print(line.rstrip())
        if 'Registered tunnel' in line or 'Connection established' in line:
            print(f'\n✅ Tunnel READY → http://{PUBLIC_HOSTNAME}')
            break

threading.Thread(target=_run_cloudflared, daemon=True).start()
time.sleep(6)

proc = globals().get('_cf_proc')
if proc and proc.poll() is None:
    print(f'[INFO] Tunnel actif (PID={proc.pid})')
    print(f'[INFO] POST → http://{PUBLIC_HOSTNAME}/infer')
    print(f'[INFO] POST → http://{PUBLIC_HOSTNAME}/fgsm')
    print(f'[INFO] POST → http://{PUBLIC_HOSTNAME}/enroll')
    print(f'[INFO] GET  → http://{PUBLIC_HOSTNAME}/health')
else:
    print('[WARN] Tunnel non démarré — vérifie les logs')

[INFO] Port 9999 libéré
[INFO] Device: cuda
Chargement de FaceNet (InceptionResnetV1) sur cuda...
[INFO] FaceRecognizer chargé
[INFO] FGSMAttacker chargé
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:9999
 * Running on http://172.28.0.12:9999
INFO:werkzeug:Press CTRL+C to quit


[INFO] Flask démarré sur le port 9999
2026-05-24T09:29:27Z INF Starting tunnel tunnelID=82d8c8e1-fe10-478e-9a97-9e01fff0ed0a
2026-05-24T09:29:27Z INF Version 2026.5.0 (Checksum 0095e46fdc88855d801c4d304cb1f5dd4bd656116c47ab94c2ad0ae7cda1c7ec)
2026-05-24T09:29:27Z INF GOOS: linux, GOVersion: go1.26.2, GoArch: amd64
2026-05-24T09:29:27Z INF Settings: map[token:***** url:http://localhost:9999]
2026-05-24T09:29:27Z INF cloudflared will not automatically update if installed by a package manager.
2026-05-24T09:29:27Z INF Generated Connector ID: a0e0d854-b39c-4340-b51f-2d1eaa32c48d
2026-05-24T09:29:27Z INF Initial protocol quic
2026-05-24T09:29:27Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-24T09:29:27Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-24T09:29:27Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-24T09:29:27Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-24T09:29:27Z INF Starting metrics server on 127.0.0.1

In [2]:
# ═══════════════════════════════════════════════
# 6️⃣  MONITORING DES REQUÊTES (optionnel)
# ═══════════════════════════════════════════════
import time as _time

_original_infer = app.view_functions['infer']
_count = 0

def _monitored_infer():
    global _count
    _count += 1
    print(f'[{_count}] Requête /infer reçue à {_time.strftime("%H:%M:%S")}')
    return _original_infer()

app.view_functions['infer'] = _monitored_infer
print('✅ Monitoring actif — chaque requête /infer sera loggée ici')

✅ Monitoring actif — chaque requête /infer sera loggée ici
